# v9 rápido – quitar el atajo "superficie lisa = Rojo" (~45 min, a prueba de cortes)
**Problema de v6/v8b:** una mesa, pared o fondo liso vacíos → "Rojo 70-98%". Causa: en *Medical Waste 4.0* los objetos
salen pequeños sobre mesas lisas.

**Qué hace (rápido):**
1. Datos: partición congelada (en Drive) + TrashNet + Waste Classification "O" + Medical Waste 4.0 **recortado al objeto**.
   Sin las fotos del examen (nombre, sesión ±20 s, pHash ≤12). Sin los datasets pesados (biomédico 5 GB, frutas 3.6 GB).
2. 30% del train de **todos** los colores se copia con el objeto pequeño sobre mesas/paredes lisas.
3. Ajuste fino corto **desde la mezcla de v8b** (10 épocas).
4. Pruebas: val congelado (nadie lo entrenó) a color, en grises y **con fondo**; **fondos vacíos**; **falsos Rojo**; cámara; examen.

**A prueba de cortes:** el dataset preparado se guarda en Drive (`v9r_dataset.zip`) y el entrenamiento guarda cada época.
Si Colab se corta: **Entorno de ejecución → Ejecutar todo** y sigue donde se quedó (no repite descargas ni épocas).

GPU T4. Primera vez ~45-60 min; si se corta, lo que falta.

In [ ]:
!pip install -q "ultralytics>=8.3,<8.4" imagehash kagglehub
from google.colab import drive
drive.mount("/content/drive")
import os, re, json, shutil, zipfile, random, hashlib, urllib.request
from collections import defaultdict, Counter
from datetime import datetime
import numpy as np, pandas as pd, torch, cv2, imagehash
from PIL import Image, ImageOps

DRIVE_DIR = "/content/drive/MyDrive/data basurita"
MODELOS, RUNS, REV = f"{DRIVE_DIR}/modelos", f"{DRIVE_DIR}/runs_clasificador", f"{DRIVE_DIR}/revision_v9r"
ZIP_V1, ZIP_EXAMEN = f"{DRIVE_DIR}/dataset_ntp900058.zip", f"{DRIVE_DIR}/examen_test.zip"
CACHE = f"{RUNS}/v9r_dataset.zip"                     # dataset preparado (para no repetir descargas)
MODELO_V5 = f"{MODELOS}/clasificador_v5.pt"
MEZCLA30 = f"{DRIVE_DIR}/revision_v8_mezcla/mezcla30.pt"   # v8b sin ajuste = punto de partida
AJUSTE_V8B = {"Amarillo": 0.25, "Azul": 0.50, "Gris": 0.50, "Marron": 0.00, "Negro": -1.00, "Rojo": 1.75}
DS = "/content/ds"
SEMILLA, PCT_GRIS, PCT_FONDO, LIM_AJUSTE, LADO_MAX = 42, 0.40, 0.30, 0.50, 512
TOPE = {"trashnet": 2500, "techsash_O": 1500, "medical4.0": 2000}
EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
NOMBRES = {"Azul": "azul_papel_carton", "Amarillo": "amarillo_metales", "Blanco": "blanco_plasticos",
           "Gris": "gris_vidrio", "Marron": "marron_organicos", "Rojo": "rojo_peligrosos", "Negro": "negro_generales"}
LARGO_A_CORTO = {v: k for k, v in NOMBRES.items()}
imagenes = lambda d: sorted(f for f in os.listdir(d) if f.lower().endswith(EXT)) if os.path.isdir(d) else []
random.seed(SEMILLA)
for r in (ZIP_V1, ZIP_EXAMEN, MODELO_V5, MEZCLA30):
    assert os.path.exists(r), f"Falta en Drive: {r}"
os.makedirs(REV, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU (Entorno -> T4)")

## 1. Examen (solo para medir y para excluir sus fotos del entrenamiento)

In [ ]:
GRUPOS = {
    "cardboard": ("garbage", "Azul (papel/cartón)", {"Azul"}, True), "paper": ("garbage", "Azul (papel/cartón)", {"Azul"}, True),
    "metal": ("garbage", "Amarillo (metal)", {"Amarillo"}, True), "plastic": ("garbage", "Blanco (plástico)", {"Blanco"}, True),
    "glass": ("garbage", "Gris (vidrio)", {"Gris"}, True), "organik": ("garbage_v4", "Marrón (Organik)", {"Marron"}, True),
    "anorganik": ("garbage_v4", "Anorganik", {"Amarillo", "Azul", "Blanco", "Gris"}, False),
    "residu": ("garbage_v4", "Residu (Negro/Rojo)", {"Negro", "Rojo"}, False),
    "infectious": ("medico", "Rojo (biocontaminado)", {"Rojo"}, True),
    "general": ("medico", "No-Rojo (médico general)", {"Amarillo", "Azul", "Blanco", "Gris", "Marron", "Negro"}, False)}
shutil.rmtree("/content/examen", ignore_errors=True)
zipfile.ZipFile(ZIP_EXAMEN).extractall("/content/examen")
filas = []
for raiz, _, archivos in os.walk("/content/examen"):
    c = os.path.basename(raiz).lower()
    if "test" in raiz.replace("\\", "/").split("/") and c in GRUPOS:
        filas += [(os.path.join(raiz, f), *GRUPOS[c], c) for f in archivos if f.lower().endswith(EXT)]
EXAMEN = pd.DataFrame(filas, columns=["ruta", "dataset", "grupo", "colores_ok", "estricta", "clase"])
print(f"Examen: {len(EXAMEN)} imagenes (deben ser 1455)")

def tallo(nombre):
    b = os.path.basename(nombre)
    m = re.match(r"(.+?)_(jpg|jpeg|png)\.rf\.", b, re.I)
    b = m.group(1) if m else os.path.splitext(b)[0]
    return re.sub(r"[^a-z0-9]", "", b.lower())

def marca_ms(nombre):
    m = re.search(r"(?<!\d)(1[5-7]\d{11})(?!\d)", os.path.basename(nombre))
    return int(m.group(1)) if m else None

def huella(ruta):
    try:
        with Image.open(ruta) as im:
            return imagehash.phash(ImageOps.exif_transpose(im).convert("RGB"))
    except Exception:
        return None

TALLOS_EX = {tallo(r) for r in EXAMEN.ruta}
TIEMPOS_EX = np.array(sorted({t for t in map(marca_ms, EXAMEN.ruta) if t}), dtype=np.int64)

## 2. Dataset: se restaura de Drive si ya existe; si no, se arma y se guarda

In [ ]:
def recortar_objeto(im):
    # recorta al objeto cuando esta sobre una superficie lisa (Medical Waste 4.0)
    w0, h0 = im.size
    g = np.asarray(im.convert("L").resize((160, max(1, int(160 * h0 / w0)))), dtype=np.float32)
    borde = np.concatenate([g[:4].ravel(), g[-4:].ravel(), g[:, :4].ravel(), g[:, -4:].ravel()])
    mascara = (np.abs(cv2.GaussianBlur(g, (5, 5), 0) - np.median(borde)) > 25).astype(np.uint8)
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    ys, xs = np.nonzero(mascara)
    if len(xs) < 0.01 * mascara.size:
        return im.crop((int(w0 * .2), int(h0 * .2), int(w0 * .8), int(h0 * .8)))
    x1, x2 = np.percentile(xs, [2, 98]); y1, y2 = np.percentile(ys, [2, 98])
    if (x2 - x1) * (y2 - y1) > 0.8 * mascara.size:
        return im
    cx, cy, lado, f = (x1 + x2) / 2, (y1 + y2) / 2, max(x2 - x1, y2 - y1) * 1.3 / 2, w0 / 160
    caja = (max(0, int((cx - lado) * f)), max(0, int((cy - lado) * f)), min(w0, int((cx + lado) * f)), min(h0, int((cy + lado) * f)))
    return im.crop(caja) if caja[2] - caja[0] > 32 and caja[3] - caja[1] > 32 else im

def fondo_aleatorio(lado, rng):
    color = rng.integers(0, 256, 3).astype(np.float32)
    tipo = rng.integers(4)
    yy, xx = np.mgrid[0:lado, 0:lado].astype(np.float32) / lado
    if tipo == 0:
        a = np.ones((lado, lado, 1)) * color + rng.normal(0, 6, (lado, lado, 3))
    elif tipo == 1:
        a = color * (0.6 + 0.6 * (xx if rng.random() < .5 else yy)[..., None])
    elif tipo == 2:
        a = color * (0.85 + 0.15 * np.sin(yy * rng.uniform(20, 60) + np.sin(xx * 6) * 2)[..., None]) + rng.normal(0, 4, (lado, lado, 3))
    else:
        a = np.full((lado, lado, 3), rng.uniform(40, 220)) + rng.normal(0, 5, (lado, lado, 3))
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

def sobre_fondo(im, rng, lado=320):
    lienzo = fondo_aleatorio(lado, rng)
    esc = rng.uniform(0.35, 0.70)
    w = int(lado * esc); h = int(w * im.height / max(1, im.width))
    if h > lado * 0.9:
        h = int(lado * 0.9); w = int(h * im.width / max(1, im.height))
    pieza = im.resize((max(16, w), max(16, h)))
    lienzo.paste(pieza, (int(rng.integers(0, lado - pieza.width + 1)), int(rng.integers(0, lado - pieza.height + 1))))
    return lienzo

def guardar(origen, destino, recortar=False):
    with Image.open(origen) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        if recortar:
            im = recortar_objeto(im)
        im.thumbnail((LADO_MAX, LADO_MAX))
        im.save(destino, "JPEG", quality=90)

if os.path.exists(CACHE):
    print("Dataset ya preparado en Drive -> se restaura (sin descargas)")
    shutil.rmtree(DS, ignore_errors=True)
    zipfile.ZipFile(CACHE).extractall("/content")
else:
    # 2.1 particion congelada (train/val/test)
    shutil.rmtree(DS, ignore_errors=True)
    zipfile.ZipFile(ZIP_V1).extractall(DS)
    for sp in ("train", "val", "test"):
        for corto, largo in NOMBRES.items():
            if os.path.isdir(f"{DS}/{sp}/{corto}"):
                os.rename(f"{DS}/{sp}/{corto}", f"{DS}/{sp}/{largo}")
    # 2.2 fuentes nuevas (solo a TRAIN; val = val congelado, que nadie entreno)
    cand = defaultdict(list)                                  # fuente -> [(ruta, color, categoria)]
    try:
        urllib.request.urlretrieve("https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip", "/content/tn.zip")
        zipfile.ZipFile("/content/tn.zip").extractall("/content/trashnet")
        MAPEO_TN = {"cardboard": "Azul", "paper": "Azul", "metal": "Amarillo", "plastic": "Blanco", "glass": "Gris", "trash": "Negro"}
        for raiz, _, arch in os.walk("/content/trashnet"):
            cat = os.path.basename(raiz).lower()
            if "__MACOSX" not in raiz and cat in MAPEO_TN:
                cand["trashnet"] += [(os.path.join(raiz, f), MAPEO_TN[cat], cat) for f in sorted(arch) if f.lower().endswith(EXT)]
    except Exception as e:
        print("AVISO TrashNet:", e)
    try:
        import kagglehub
        ruta_wc, vistos = kagglehub.dataset_download("techsash/waste-classification-data"), set()
        for raiz, _, arch in os.walk(ruta_wc):
            if os.path.basename(raiz).upper() == "O":
                for f in sorted(arch):
                    if f.lower().endswith(EXT) and f not in vistos:
                        vistos.add(f); cand["techsash_O"].append((os.path.join(raiz, f), "Marron", "O"))
    except Exception as e:
        print("AVISO Waste Classification:", e)
    MW_DIR = "/content/medical_waste"
    try:
        if not os.path.isdir(MW_DIR):
            print("Descargando Medical Waste 4.0 (1-2 GB)...")
            urllib.request.urlretrieve("https://zenodo.org/records/7643417/files/Medical%20Waste%204.0.zip?download=1", "/content/mw.zip")
            zipfile.ZipFile("/content/mw.zip").extractall(MW_DIR); os.remove("/content/mw.zip")
        TIPOS_MW = ("gauze", "glove", "cap", "glasses", "shoe", "tube", "urine")
        for raiz, _, arch in os.walk(MW_DIR):
            if "__MACOSX" in raiz:
                continue
            for f in sorted(arch):
                if f.lower().endswith(EXT):
                    tipo = next((t for t in TIPOS_MW if t in raiz.lower()), "otros")
                    vista = "estereo" if re.search(r"_(l|r)\.", f.lower()) else "rgb"
                    cand["medical4.0"].append((os.path.join(raiz, f), "Rojo", f"{tipo}_{vista}"))
    except Exception as e:
        print("AVISO Medical Waste 4.0:", e)

    # 2.3 quitar fotos del examen (nombre / sesion +-20 s) y elegir balanceado por categoria
    def es_del_examen(r):
        if tallo(r) in TALLOS_EX:
            return True
        t = marca_ms(r)
        return t is not None and len(TIEMPOS_EX) and int(np.abs(TIEMPOS_EX - t).min()) <= 20000
    elegidas, quitadas = [], Counter()
    for fuente, items in cand.items():
        limpios = [it for it in items if not es_del_examen(it[0])]
        quitadas[fuente] += len(items) - len(limpios)
        por_cat = defaultdict(list)
        for it in limpios:
            por_cat[it[2]].append(it)
        listas = []
        for cat in sorted(por_cat):
            l = sorted(por_cat[cat]); random.Random(f"{SEMILLA}|{fuente}|{cat}").shuffle(l); listas.append(l)
        sel = []
        while len(sel) < TOPE[fuente] and any(listas):
            for l in listas:
                if l and len(sel) < TOPE[fuente]:
                    sel.append(l.pop())
        elegidas += [(r, c, fuente) for r, c, _ in sel]
    print("Quitadas por nombre/sesion del examen:", dict(quitadas))

    # 2.4 quitar casi-duplicados del examen (pHash <= 12)
    H_EX = [h for h in map(huella, EXAMEN.ruta) if h is not None]
    H_EX_INT = np.array([int(str(h), 16) for h in H_EX], dtype=np.uint64)
    def cerca_del_examen(h):
        d = np.unpackbits(np.bitwise_xor(H_EX_INT, np.uint64(int(str(h), 16))).view(np.uint8)).reshape(-1, 64).sum(1)
        return d.min() <= 12
    agregadas, por_phash = Counter(), 0
    for i, (ruta, color, fuente) in enumerate(elegidas):
        h = huella(ruta)
        if h is None:
            continue
        if cerca_del_examen(h):
            por_phash += 1; continue
        largo = NOMBRES[color]
        destino = f"{DS}/train/{largo}/v9r_{fuente}_{hashlib.md5(ruta.encode()).hexdigest()[:10]}.jpg"
        try:
            guardar(ruta, destino, recortar=(fuente == "medical4.0"))
            agregadas[(fuente, color)] += 1
        except Exception as e:
            print("omitida", ruta, e)
    print("Quitadas por parecerse al examen (pHash):", por_phash)
    print(pd.Series(agregadas).rename("agregadas_a_train").to_string())

    # 2.5 grises (40% del train) y copias sobre fondo liso (30% del train, todos los colores)
    rng = np.random.default_rng(SEMILLA)
    for c in NOMBRES.values():
        fotos = imagenes(f"{DS}/train/{c}")
        for f in sorted(random.Random(f"{SEMILLA}|gris|{c}").sample(fotos, int(len(fotos) * PCT_GRIS))):
            p = f"{DS}/train/{c}/{f}"
            Image.open(p).convert("L").convert("RGB").save(p, "JPEG", quality=92)
        for f in fotos[::max(1, round(1 / PCT_FONDO))]:
            with Image.open(f"{DS}/train/{c}/{f}") as im:
                sobre_fondo(im.convert("RGB"), rng).save(f"{DS}/train/{c}/fondo_{f}", "JPEG", quality=90)
    # 2.6 guardar en Drive para no repetir nada si Colab se corta
    print("Guardando el dataset preparado en Drive (unos minutos)...")
    shutil.make_archive("/content/v9r_dataset", "zip", "/content", "ds")
    shutil.copy("/content/v9r_dataset.zip", CACHE)

print({sp: {c: len(imagenes(f"{DS}/{sp}/{c}")) for c in NOMBRES.values()} for sp in ("train", "val", "test")})

# Pruebas (se generan siempre igual): val en grises, val con fondo, fondos vacios
VAL_GRIS, VAL_FONDO, VACIOS_DIR = "/content/val_gris", "/content/val_fondo", "/content/fondos_vacios"
for d in (VAL_GRIS, VAL_FONDO, VACIOS_DIR):
    shutil.rmtree(d, ignore_errors=True)
rng_v = np.random.default_rng(SEMILLA + 1)
for c in NOMBRES.values():
    os.makedirs(f"{VAL_GRIS}/{c}"); os.makedirs(f"{VAL_FONDO}/{c}")
    for f in imagenes(f"{DS}/val/{c}"):
        with Image.open(f"{DS}/val/{c}/{f}") as im:
            im = im.convert("RGB")
            im.convert("L").convert("RGB").save(f"{VAL_GRIS}/{c}/{f}")
            sobre_fondo(im, rng_v).save(f"{VAL_FONDO}/{c}/{f}", "JPEG", quality=90)
os.makedirs(VACIOS_DIR)
for i in range(300):
    fondo_aleatorio(320, rng_v).save(f"{VACIOS_DIR}/fondo_{i:03d}.jpg", "JPEG", quality=90)
CAMARA = "/content/test_camara"
shutil.rmtree(CAMARA, ignore_errors=True)
for corto, largo in NOMBRES.items():
    os.makedirs(f"{CAMARA}/{largo}")
    for f in imagenes(f"{DRIVE_DIR}/test_camara/{corto}"):
        shutil.copy2(f"{DRIVE_DIR}/test_camara/{corto}/{f}", f"{CAMARA}/{largo}/{f}")

## 3. Ajuste fino desde v8b (10 épocas). Si se cortó, continúa; si terminó, se salta.

In [ ]:
from ultralytics import YOLO
V9R, ULTIMO = f"{RUNS}/v9r/weights/best.pt", f"{RUNS}/v9r/weights/last.pt"
if os.path.exists(ULTIMO) and torch.load(ULTIMO, map_location="cpu", weights_only=False).get("epoch") == -1:
    print("v9r ya estaba entrenado -> se reutiliza")
elif os.path.exists(ULTIMO):
    print("Entrenamiento a medias -> se continua desde la ultima epoca")
    YOLO(ULTIMO).train(resume=True)
else:
    YOLO(MEZCLA30).train(data=DS, epochs=10, imgsz=224, batch=64, optimizer="SGD", lr0=0.0005, lrf=0.2,
                         warmup_epochs=0, patience=4, seed=SEMILLA, project=RUNS, name="v9r", exist_ok=True)

## 4. Pruebas y elección (con val congelado: nadie lo entrenó)

In [ ]:
from sklearn.metrics import f1_score

def hornear_dict(ruta, ajuste, salida):
    ck = torch.load(ruta, map_location="cpu", weights_only=False)
    m = (ck["ema"] if ck.get("ema") is not None else ck["model"]).float()
    for i, n in m.names.items():
        m.model[-1].linear.bias.data[i] += ajuste.get(LARGO_A_CORTO[n], 0.0)
    ck["model"], ck["ema"] = m.half(), None
    torch.save(ck, salida)
    return salida

def probabilidades(ruta_modelo, rutas):
    m = YOLO(ruta_modelo)
    orden = [LARGO_A_CORTO[m.names[i]] for i in range(len(m.names))]
    rutas = list(rutas); P = []
    for i in range(0, len(rutas), 128):
        P += [r.probs.data.cpu().numpy() for r in m.predict(rutas[i:i + 128], imgsz=224, verbose=False, stream=True)]
    return np.array(P), orden

def rutas_de(carpeta):
    rr, yy = [], []
    for c in sorted(os.listdir(carpeta)):
        for f in imagenes(f"{carpeta}/{c}"):
            rr.append(f"{carpeta}/{c}/{f}"); yy.append(LARGO_A_CORTO[c])
    return rr, np.array(yy)

def balanceado(y, p):
    return float(np.mean([(p[y == c] == c).mean() for c in np.unique(y)]))

def con_sesgos(P, orden, b):
    return np.array(orden)[(P * np.exp(b)).argmax(1)]

VAL = rutas_de(f"{DS}/val"); VALG = rutas_de(VAL_GRIS); VALF = rutas_de(VAL_FONDO)
TEST = rutas_de(f"{DS}/test"); CAM = rutas_de(CAMARA)
VACIOS = [f"{VACIOS_DIR}/{f}" for f in imagenes(VACIOS_DIR)]
V1I = EXAMEN[EXAMEN.dataset == "garbage"]
os.makedirs("/content/_gris_v1i", exist_ok=True); RUT_G = []
for i, r in enumerate(V1I.ruta):
    d = f"/content/_gris_v1i/{i}.jpg"; Image.open(r).convert("L").convert("RGB").save(d); RUT_G.append(d)

def hora(n):
    m = re.search(r"(\d{8}_\d{6})", n)
    return datetime.strptime(m.group(1), "%Y%m%d_%H%M%S") if m else None

def agrupar_por_hora(fotos):
    fotos = sorted(fotos, key=lambda f: (hora(f) or datetime.min, f)); grupos, prev = [], None
    for f in fotos:
        t = hora(f)
        if prev is None or t is None or (t - prev).total_seconds() > 60:
            grupos.append([])
        grupos[-1].append(f); prev = t
    return grupos

def objetos_camara(preds):
    df, ok = pd.DataFrame({"n": [os.path.basename(r) for r in CAM[0]], "y": CAM[1], "p": preds}), []
    for c, g in df.groupby("y"):
        for grupo in agrupar_por_hora(list(g.n)):
            ok.append(g[g.n.isin(grupo)].p.mode()[0] == c)
    return f"{sum(ok)}/{len(ok)}"

def evaluar(nombre, ruta):
    P = {k: probabilidades(ruta, x)[0] for k, x in (("val", VAL[0]), ("valg", VALG[0]), ("valf", VALF[0]),
         ("test", TEST[0]), ("cam", CAM[0]), ("vac", VACIOS), ("ex", EXAMEN.ruta), ("g", RUT_G))}
    orden = probabilidades(ruta, VAL[0][:1])[1]
    return P, orden

def medir(P, orden, b):
    pr = {k: con_sesgos(v, orden, b) for k, v in P.items()}
    ex = EXAMEN.assign(pred=pr["ex"]); ex["ok"] = [p in o for p, o in zip(ex.pred, ex.colores_ok)]
    grupos = ex.groupby("grupo").ok.mean()
    est = ex[ex.estricta]; y = est.colores_ok.map(lambda s: next(iter(s)))
    mf1 = f1_score(y, est.pred, labels=sorted(y.unique()), average="macro", zero_division=0)
    med = ex[ex.dataset == "medico"]
    rec = (med[med.clase == "infectious"].pred == "Rojo").mean()
    rojos = med[med.pred == "Rojo"]; prec = (rojos.clase == "infectious").mean() if len(rojos) else 0
    f1r = 2 * prec * rec / (prec + rec) if prec + rec else 0
    org = ex[ex.clase == "organik"].ok.mean()
    caida = (ex[ex.dataset == "garbage"].ok.mean() - np.mean([p in o for p, o in zip(pr["g"], V1I.colores_ok)])) * 100
    base = 8 * grupos.mean() + 4 * mf1 + 3 + (3 - 0.05 * caida); seg = 3 * org + 28 / 9 * f1r
    fila = {"val_color": balanceado(VAL[1], pr["val"]), "val_grises": balanceado(VALG[1], pr["valg"]),
            "val_con_fondo": balanceado(VALF[1], pr["valf"]),
            "fondos_vacios_dice_Rojo": float((pr["vac"] == "Rojo").mean()),
            "fondos_vacios_mas_comun": pd.Series(pr["vac"]).value_counts().index[0],
            "falsos_Rojo_con_fondo": float((pr["valf"][VALF[1] != "Rojo"] == "Rojo").mean()),
            "test_viejo": float((pr["test"] == TEST[1]).mean()), "camara_fotos": float((pr["cam"] == CAM[1]).mean()),
            "camara_objetos": objetos_camara(pr["cam"]), **grupos.round(3).to_dict(),
            "nota": round(base + seg, 2), "nota_seg_max2": round(base + min(seg, 2), 2)}
    fila["puntaje"] = (fila["val_color"] + fila["val_grises"] + fila["val_con_fondo"]) / 3
    return fila

V8B = hornear_dict(MEZCLA30, AJUSTE_V8B, f"{MODELOS}/clasificador_v8b.pt")
CANDIDATOS = {"v5": MODELO_V5, "v8b": V8B, "v9r": V9R}
FILAS, SESGOS = {}, {}
for nombre, ruta in list(CANDIDATOS.items()):
    P, orden = evaluar(nombre, ruta)
    FILAS[nombre] = medir(P, orden, np.zeros(len(orden)))
    if nombre == "v9r":                                      # ajuste moderado (+-0.5) con val color+grises+fondo
        obj = lambda b: (balanceado(VAL[1], con_sesgos(P["val"], orden, b)) + balanceado(VALG[1], con_sesgos(P["valg"], orden, b))
                         + balanceado(VALF[1], con_sesgos(P["valf"], orden, b))) / 3
        b = np.zeros(len(orden)); mejor = obj(b)
        for _ in range(3):
            cambio = False
            for k in range(len(orden)):
                for v in np.arange(-LIM_AJUSTE, LIM_AJUSTE + .01, .25):
                    bb = b.copy(); bb[k] = v
                    if obj(bb) > mejor + 1e-4:
                        mejor, b, cambio = obj(bb), bb, True
            if not cambio:
                break
        if np.any(b != 0):
            FILAS["v9r_cal"] = medir(P, orden, b)
            SESGOS["v9r_cal"] = dict(zip(orden, b))
            CANDIDATOS["v9r_cal"] = hornear_dict(ruta, SESGOS["v9r_cal"], "/content/v9r_cal.pt")
            print("Ajuste v9r_cal:", {k: v for k, v in SESGOS["v9r_cal"].items() if v})
TABLA = pd.DataFrame(FILAS)
pd.set_option("display.width", 200)
print(TABLA.to_string())
TABLA.to_csv(f"{REV}/comparacion.csv")

## 5. Decisión (regla fija, contra v8b)

In [ ]:
ELEGIDO = max((k for k in FILAS), key=lambda k: FILAS[k]["puntaje"])
e, o = FILAS[ELEGIDO], FILAS["v8b"]
reglas = {"mejor val (color+grises+fondo)": e["puntaje"] > o["puntaje"],
          "menos fondos vacios = Rojo": e["fondos_vacios_dice_Rojo"] < o["fondos_vacios_dice_Rojo"],
          "no mas falsos Rojo": e["falsos_Rojo_con_fondo"] <= o["falsos_Rojo_con_fondo"],
          "test viejo sin bajar >2 pts": e["test_viejo"] >= o["test_viejo"] - 0.02,
          "camara sin perder >1 objeto": int(e["camara_objetos"].split("/")[0]) >= int(o["camara_objetos"].split("/")[0]) - 1}
print("Elegido por val:", ELEGIDO)
for k, v in reglas.items():
    print(f"  {'OK ' if v else 'NO '} {k}")
if ELEGIDO.startswith("v9r"):
    shutil.copy(CANDIDATOS[ELEGIDO], f"{MODELOS}/clasificador_v9.pt")
if ELEGIDO.startswith("v9r") and all(reglas.values()):
    shutil.copy(CANDIDATOS[ELEGIDO], f"{MODELOS}/clasificador.pt")
    print(f"\n{ELEGIDO} GANA -> modelos/clasificador.pt ahora es v9 (copia en clasificador_v9.pt)")
else:
    print("\nNo cumple todas las reglas -> clasificador.pt no cambia")
print("md5 de clasificador.pt:", hashlib.md5(open(f"{MODELOS}/clasificador.pt", "rb").read()).hexdigest())